# Data Pipeline Overview


## Table of Contents

1. [Setup](#1-setup)
2. [Data Collection](#2-data-collection)
   - 2a. Task descriptions
   - 2b. LLM-generated code samples
   - 2c. Pass/fail evaluation results
3. [Preprocessing](#3-preprocessing)
   - 3a. Model name normalization
   - 3b. Matching samples to labels
   - 3c. Merging task metadata
4. [EDA](#4-eda)
   - 4a. Dataset overview
   - 4b. Label distribution
   - 4c. Model performance
   - 4d. Solution characteristics
5. [Export](#5-export)
6. [Pipeline Summary](#6-pipeline-summary)

## 1. Setup <a id="1-setup"></a>

In [1]:
import json, re, zipfile
from pathlib import Path

import pandas as pd
import requests
from datasets import load_dataset, load_dataset_builder
from tqdm.notebook import tqdm

RAW = Path("raw")
PROCESSED = Path("processed")
TASKS_PATH = RAW / "bigcodebench_tasks.jsonl"
SAMPLES_ZIP = RAW / "sanitized_calibrated_samples.zip"
SAMPLES_DIR = RAW / "sanitized_calibrated_samples"
EVAL_DIR = RAW / "eval_results"

for d in [RAW, PROCESSED, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 2. Data Collection <a id="2-data-collection"></a>

Three sources, each downloaded once and cached locally:
- **Tasks**: 1,140 Python programming problems from BigCodeBench on HuggingFace
- **Samples**: code generated by 100+ LLMs, published as a zip on GitHub
- **Labels**: per-model pass/fail results from two HuggingFace performance datasets

### 2a. Task descriptions

In [2]:
if TASKS_PATH.exists():
    print("Tasks already downloaded.")
else:
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    ds.to_json(str(TASKS_PATH))
    print(f"Saved {len(ds)} tasks.")

Tasks already downloaded.


In [3]:
tasks = {}
with open(TASKS_PATH) as f:
    for line in f:
        t = json.loads(line)
        tasks[t["task_id"]] = t

tasks_df = pd.DataFrame([
    {"task_id": t["task_id"], "entry_point": t["entry_point"], "libs": t["libs"],
     "complete_prompt_len": len(t.get("complete_prompt", "")),
     "instruct_prompt_len": len(t.get("instruct_prompt", ""))}
    for t in tasks.values()
])
print(f"{len(tasks_df)} tasks loaded.")
tasks_df.head()

1140 tasks loaded.


,task_id,entry_point,libs,complete_prompt_len,instruct_prompt_len
0,BigCodeBench/0,task_func,"['random', 'itertools']",671,559
1,BigCodeBench/1,task_func,"['collections', 'random', 'string']",888,581
2,BigCodeBench/2,task_func,"['statistics', 'random']",885,483
3,BigCodeBench/3,task_func,"['numpy', 'random']",1026,543
4,BigCodeBench/4,task_func,"['collections', 'itertools']",930,626


### 2b. LLM-generated code samples

158 MB zip from BigCodeBench GitHub releases (v0.2.5). We only use `complete/` and `instruct/` — the `full/` subdirectory has newer models without eval labels, and `hard/` only covers 148 tasks.

In [4]:
if SAMPLES_DIR.exists() and any(SAMPLES_DIR.glob("**/*.jsonl")):
    print("Samples already extracted.")
else:
    if not SAMPLES_ZIP.exists():
        url = "https://github.com/bigcode-project/bigcodebench/releases/download/v0.2.5/sanitized_calibrated_samples.zip"
        resp = requests.get(url, stream=True, timeout=300)
        resp.raise_for_status()
        total = int(resp.headers.get("content-length", 0))
        with open(SAMPLES_ZIP, "wb") as fh, tqdm(total=total, unit="B", unit_scale=True) as bar:
            for chunk in resp.iter_content(1 << 20):
                fh.write(chunk)
                bar.update(len(chunk))
    with zipfile.ZipFile(SAMPLES_ZIP) as zf:
        zf.extractall(RAW)
    print("Done.")

for subdir in sorted(SAMPLES_DIR.iterdir()):
    if subdir.is_dir():
        print(f"  {subdir.name}/: {len(list(subdir.glob('**/*.jsonl')))} model files")

Samples already extracted.
  complete/: 87 model files
  full/: 250 model files
  hard/: 101 model files
  instruct/: 134 model files


### 2c. Pass/fail evaluation results

Downloaded from `bigcode/bigcodebench-complete-perf` (85 models) and `bigcode/bigcodebench-instruct-perf` (52 models). Each model split has rows of `{task_id, status}` where status is 0 or 1.

In [5]:
existing = list(EVAL_DIR.glob("*.json"))
if len(existing) >= 137:
    print(f"Eval results already downloaded ({len(existing)} files).")
else:
    for ds_name, tag in [("bigcode/bigcodebench-complete-perf", "complete"),
                         ("bigcode/bigcodebench-instruct-perf", "instruct")]:
        builder = load_dataset_builder(ds_name)
        for model_split in tqdm(list(builder.info.splits.keys()), desc=tag):
            out = EVAL_DIR / f"{model_split}--{tag}_eval_results.json"
            if out.exists():
                continue
            ds = load_dataset(ds_name, split=model_split)
            with open(out, "w") as fh:
                json.dump({r["task_id"]: int(r["status"]) for r in ds}, fh)
    print(f"Downloaded {len(list(EVAL_DIR.glob('*.json')))} eval files.")

Eval results already downloaded (137 files).


## 3. Preprocessing <a id="3-preprocessing"></a>

The main challenge is that sample filenames and eval result names use different conventions for the same model. We normalize names, match them up, and merge everything into one flat CSV.

### 3a. Model name normalization

Fuzzy normalization handles most cases. A manual mapping covers edge cases like API-style names, `-hf` suffixes, and version tags.

In [6]:
MANUAL_MAP = {
    "codellama--CodeLlama-7b-Instruct-hf":        "CodeLlama_7B_Instruct",
    "codellama--CodeLlama-13b-Instruct-hf":       "CodeLlama_13B_Instruct",
    "codellama--CodeLlama-34b-Instruct-hf":       "CodeLlama_34B_Instruct",
    "codellama--CodeLlama-70b-Instruct-hf":       "CodeLlama_70B_Instruct",
    "mistralai--Mixtral-8x22B-Instruct-v0.1":     "Mixtral_8x22B_Instruct",
    "mistralai--Mistral-7B-Instruct-v0.3":        "Mistral_7B_Instruct_v0.3",
    "gemini-1.5-flash":                           "Gemini_1.5_Flash_API_0514",
    "gemini-1.5-pro":                             "Gemini_1.5_Pro_API_0514",
    "google--codegemma-7b-it":                    "CodeGemma_7B_Instruct",
    "google--gemma-2-9b-it":                      "Gemma_2_9B_Instruct",
    "meta-llama--Meta-Llama-3-8B-Instruct":       "Llama_3_8B_Instruct",
    "meta-llama--Meta-Llama-3-70B-Instruct":      "Llama_3_70B_Instruct",
    "meta-llama--Meta-Llama-3.1-8B-Instruct":     "Llama_3_8B_Instruct",
    "meta-llama--Meta-Llama-3.1-70B-Instruct":    "Llama_3_70B_Instruct",
    "bigcode--starcoder2-15b-instruct-v0.1":      "StarCoder2_15B_Instruct_v0.1",
    "CohereForAI--c4ai-command-r-plus":           "Command_R_plus",
    "ibm-granite--granite-3b-code-instruct":      "Granite_Code_3B_Instruct",
    "ibm-granite--granite-8b-code-instruct":      "Granite_Code_8B_Instruct",
    "ibm-granite--granite-20b-code-instruct":     "Granite_Code_20B_Instruct",
    "ibm-granite--granite-34b-code-instruct":     "Granite_Code_34B_Instruct",
}

def normalize(name):
    if "--" in name:
        name = name.split("--", 1)[1]
    name = re.sub(r"-hf$", "", name)
    name = re.sub(r"[-_v. ]", "", name)
    name = re.sub(r"\d{8}$", "", name)
    return name.lower()

### 3b. Matching samples to labels

In [7]:
def load_eval_lookup(split):
    lookup = {}
    for f in EVAL_DIR.glob(f"*--{split}_eval_results.json"):
        model = f.stem.replace(f"--{split}_eval_results", "")
        with open(f) as fh:
            lookup[model] = json.load(fh)
    return lookup

records = []

for split in ["complete", "instruct"]:
    sample_subdir = SAMPLES_DIR / split
    if not sample_subdir.exists():
        continue

    eval_lookup = load_eval_lookup(split)
    eval_norm = {normalize(k): (k, v) for k, v in eval_lookup.items()}

    matched, skipped = 0, 0
    for sf in tqdm(sorted(sample_subdir.glob("*.jsonl")), desc=split):
        sample_model = sf.stem.split("--bigcodebench")[0]

        if sample_model in MANUAL_MAP:
            eval_name = MANUAL_MAP[sample_model]
            labels = eval_lookup.get(eval_name)
        else:
            match = eval_norm.get(normalize(sample_model))
            eval_name, labels = match if match else (None, None)

        if labels is None:
            skipped += 1
            continue

        matched += 1
        with open(sf) as fh:
            for line in fh:
                row = json.loads(line)
                label = labels.get(row["task_id"])
                if label is None:
                    continue
                records.append({
                    "task_id": row["task_id"], "model_name": eval_name,
                    "split": split, "solution": row.get("solution", ""), "label": label,
                })

    print(f"  {split}: {matched} matched, {skipped} skipped")

print(f"\nTotal records: {len(records):,}")

complete:   0%|          | 0/87 [00:00<?, ?it/s]

  complete: 56 matched, 31 skipped


instruct:   0%|          | 0/134 [00:00<?, ?it/s]

  instruct: 54 matched, 80 skipped

Total records: 123,416


### 3c. Merging task metadata

Join prompts, required libraries, and entry points into the samples so we have one self-contained file.

In [8]:
samples_df = pd.DataFrame(records)

task_meta = pd.DataFrame([
    {"task_id": t["task_id"], "complete_prompt": t.get("complete_prompt", ""),
     "instruct_prompt": t.get("instruct_prompt", ""), "libs": t.get("libs", ""),
     "entry_point": t.get("entry_point", "")}
    for t in tasks.values()
])

samples_df = samples_df.merge(task_meta, on="task_id", how="left")
print(f"{len(samples_df):,} rows, {samples_df['model_name'].nunique()} models, {samples_df['task_id'].nunique()} tasks")
samples_df.head()

123,416 rows, 57 models, 1140 tasks


,task_id,model_name,split,solution,label,complete_prompt,instruct_prompt,libs,entry_point
0,BigCodeBench/0,Yi_1.5_34B_Chat,complete,import itertools\nfrom random import shuffle\n...,0,import itertools\nfrom random import shuffle\n...,Calculates the average of the sums of absolute...,"['random', 'itertools']",task_func
1,BigCodeBench/1,Yi_1.5_34B_Chat,complete,import collections\nimport random\nimport stri...,1,import collections\nimport random\nimport stri...,Generate a random string of the specified leng...,"['collections', 'random', 'string']",task_func
2,BigCodeBench/2,Yi_1.5_34B_Chat,complete,import random\nimport statistics\nimport rando...,1,import random\nimport statistics\n\ndef task_f...,Create a dictionary in which keys are random l...,"['statistics', 'random']",task_func
3,BigCodeBench/3,Yi_1.5_34B_Chat,complete,import random\nimport numpy as np\ndef task_fu...,1,import random\nimport numpy as np\n\ndef task_...,Create a dictionary where keys are specified l...,"['numpy', 'random']",task_func
4,BigCodeBench/4,Yi_1.5_34B_Chat,complete,from collections import Counter\nimport iterto...,1,from collections import Counter\nimport iterto...,Count the occurrence of each integer in the va...,"['collections', 'itertools']",task_func


## 4. EDA <a id="4-eda"></a>

### 4a. Dataset overview

In [9]:
print(f"Shape: {samples_df.shape}")
print(f"\nDtypes:")
print(samples_df.dtypes)
print(f"\nNull counts:")
print(samples_df.isnull().sum())

Shape: (123416, 9)

Dtypes:
task_id            object
model_name         object
split              object
solution           object
label               int64
complete_prompt    object
instruct_prompt    object
libs               object
entry_point        object
dtype: object

Null counts:
task_id            0
model_name         0
split              0
solution           0
label              0
complete_prompt    0
instruct_prompt    0
libs               0
entry_point        0
dtype: int64


### 4b. Label distribution

In [10]:
print(f"Overall pass rate: {samples_df['label'].mean():.1%}\n")

split_stats = samples_df.groupby("split").agg(
    count=("label", "size"), pass_rate=("label", "mean")
).round(3)
split_stats

Overall pass rate: 41.2%



,count,pass_rate
split,,
complete,63840,0.455
instruct,59576,0.365


### 4c. Model performance

In [11]:
model_stats = samples_df.groupby("model_name").agg(
    samples=("label", "size"), pass_rate=("label", "mean")
).sort_values("pass_rate", ascending=False).round(3)

print("Top 10:")
display(model_stats.head(10))
print("\nBottom 10:")
display(model_stats.tail(10))

Top 10:


,samples,pass_rate
model_name,,
DeepSeek_Coder_V2_Instruct,2280,0.540
GPT_4_Turbo_2024_04_09,2280,0.532
ReflectionCoder_DS_33B,1140,0.530
Codestral_22B_v0.1,1140,0.528
GPT_4o_2024_05_13,2576,0.525
GPT_4_0613,2280,0.516
Claude_3_Opus_20240229,2280,0.514
Hermes_2_Theta_Llama_3_70B,2280,0.506
Gemini_1.5_Pro_API_0514,2280,0.506



Bottom 10:


,samples,pass_rate
model_name,,
Llama_3_8B_Instruct,4560,0.344
Mistral_Large_2402,2280,0.342
CodeLlama_34B_Instruct,2280,0.323
Granite_Code_3B_Instruct,1140,0.314
CodeLlama_13B_Instruct,2280,0.301
Yi_1.5_6B_Chat,2280,0.297
DeepSeek_Coder_1.3B_Instruct,2280,0.262
OpenCodeInterpreter_DS_1.3B,1140,0.253
CodeLlama_7B_Instruct,2280,0.238


### 4d. Solution characteristics

In [12]:
samples_df["solution_len"] = samples_df["solution"].str.len()
samples_df["solution_lines"] = samples_df["solution"].str.count("\n") + 1

print("Solution length (chars):")
print(samples_df["solution_len"].describe().round(0).to_string())
print(f"\nSolution length (lines):")
print(samples_df["solution_lines"].describe().round(0).to_string())

samples_df.drop(columns=["solution_len", "solution_lines"], inplace=True)

Solution length (chars):
count    123416.0
mean       1146.0
std         591.0
min           0.0
25%         707.0
50%        1048.0
75%        1478.0
max        6861.0

Solution length (lines):
count    123416.0
mean         33.0
std          15.0
min           1.0
25%          21.0
50%          31.0
75%          43.0
max         196.0


## 5. Export <a id="5-export"></a>

In [13]:
out = PROCESSED / "samples.csv"
samples_df.to_csv(out, index=False)
print(f"Saved {len(samples_df):,} rows to {out}")

Saved 123,416 rows to processed/samples.csv


## 6. Pipeline Summary <a id="6-pipeline-summary"></a>

```
                          DATA PIPELINE FLOW

    Source 1                Source 2                 Source 3
    HuggingFace            GitHub Releases          HuggingFace
    bigcode/bigcodebench   bigcodebench v0.2.5      bigcodebench-*-perf
         |                      |                        |
         v                      v                        v
    1,140 tasks            158 MB zip               137 JSON files
    (prompts, tests,       (100+ LLMs x             (per-model
     entry points)          1,140 tasks)              pass/fail)
         |                      |                        |
         |               -------+-------                 |
         |               |             |                 |
         |               v             v                 |
         |          complete/      instruct/             |
         |          87 models      134 models            |
         |               |             |                 |
         |               +------+------+                 |
         |                      |                        |
         |                      v                        |
         |              Name Normalization                |
         |              (fuzzy + manual map)              |
         |                      |                        |
         |                      +<-----------------------+
         |                      |
         |                      v
         |              Match & Join
         |              57 models matched
         |              (others dropped: no labels)
         |                      |
         +--------------------->+
                                |
                                v
                        Merge Task Metadata
                        (prompts, libs, entry_point)
                                |
                                v
                       samples.csv
                       123,416 rows x 9 columns
                       57 models, 1,140 tasks
                       41% overall pass rate
```